In [ ]:
#imports
import numpy as np
import pandas as pd
from collections import Counter

import torch
from torchinfo import summary
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_curve,
    auc
)

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

In [ ]:
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128

meta_path = "/kaggle/input/datasets/sujansarkar/ham10000-folder-wise/HAM10000_metadata.csv"
df = pd.read_csv(meta_path)

# HAM10000 can have multiple images per lesion — split on lesion_id to prevent leakage
gss_train = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
train_idx, temp_idx = next(gss_train.split(df, groups=df['lesion_id']))

train_df = df.iloc[train_idx].copy()
temp_df  = df.iloc[temp_idx].copy()

gss_val = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df['lesion_id']))

val_df  = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

print("--- HAM10000 Tabular-Only Data Split Report ---")
print(f"Train Set: {len(train_df):>5} rows | Lesions: {train_df['lesion_id'].nunique()}")
print(f"Val Set:   {len(val_df):>5} rows | Lesions: {val_df['lesion_id'].nunique()}")
print(f"Test Set:  {len(test_df):>5} rows | Lesions: {test_df['lesion_id'].nunique()}")

print("\n--- Class Distribution ---")
print("Train:", dict(train_df['dx'].value_counts()))
print("Val:  ", dict(val_df['dx'].value_counts()))
print("Test: ", dict(test_df['dx'].value_counts()))

In [ ]:
NUMERICAL_COLS   = ['age']
CATEGORICAL_COLS = ['sex', 'localization', 'dx_type']

def preprocess_tabular(train_df, val_df, test_df):
    df_train = train_df.copy()
    df_val   = val_df.copy()
    df_test  = test_df.copy()

    # Fit imputation and scaling on train only, apply to all splits
    train_medians = {col: pd.to_numeric(df_train[col], errors='coerce').median()
                     for col in NUMERICAL_COLS}

    scalers = {}
    for col in NUMERICAL_COLS:
        for df in [df_train, df_val, df_test]:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(train_medians[col])
        sc = StandardScaler()
        df_train[f"{col}_norm"] = sc.fit_transform(df_train[[col]]).ravel()
        df_val[f"{col}_norm"]   = sc.transform(df_val[[col]]).ravel()
        df_test[f"{col}_norm"]  = sc.transform(df_test[[col]]).ravel()
        scalers[col] = sc

    for col in CATEGORICAL_COLS:
        for df in [df_train, df_val, df_test]:
            df[col] = df[col].fillna('unknown').astype(str)

    df_train = pd.get_dummies(df_train, columns=CATEGORICAL_COLS, dummy_na=False)
    df_val   = pd.get_dummies(df_val,   columns=CATEGORICAL_COLS, dummy_na=False)
    df_test  = pd.get_dummies(df_test,  columns=CATEGORICAL_COLS, dummy_na=False)

    # Align columns so val/test never have columns train hasn't seen
    df_train, df_val  = df_train.align(df_val,  join='left', axis=1, fill_value=0)
    df_train, df_test = df_train.align(df_test, join='left', axis=1, fill_value=0)

    for df in [df_train, df_val, df_test]:
        if 'image_id' in df.columns:
            df.set_index('image_id', inplace=True)

    return df_train, df_val, df_test

train_df, val_df, test_df = preprocess_tabular(train_df, val_df, test_df)

META_COLS        = [c for c in train_df.columns
                    if c.startswith(tuple(CATEGORICAL_COLS)) or c.endswith('_norm')]
NUM_META_FEATURES = len(META_COLS)

print(f"Tabular features processed. MLP Input Vector Size: {NUM_META_FEATURES}")

In [ ]:
class TabularOnlyDataset(Dataset):
    def __init__(self, df, meta_cols):
        self.df      = df
        self.meta_cols = meta_cols
        self.classes      = sorted(df['dx'].unique())
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        meta  = row[self.meta_cols].values.astype('float32')
        label = self.class_to_idx[row['dx']]
        return torch.tensor(meta), torch.tensor(label)

train_dataset = TabularOnlyDataset(train_df, META_COLS)
val_dataset   = TabularOnlyDataset(val_df,   META_COLS)
test_dataset  = TabularOnlyDataset(test_df,  META_COLS)

# Class weights for FocalLoss — inverse sqrt-frequency, normalised to mean=1
train_targets = [train_dataset.class_to_idx[dx] for dx in train_df['dx']]
class_counts  = Counter(train_targets)
NUM_CLASSES   = len(class_counts)
total_samples = sum(class_counts.values())
raw_w         = [total_samples / (NUM_CLASSES * class_counts[i] ** 0.5) for i in range(NUM_CLASSES)]
mean_w        = sum(raw_w) / NUM_CLASSES
class_weights_tensor = torch.FloatTensor([w / mean_w for w in raw_w])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

class_to_idx = train_dataset.class_to_idx
IDX_TO_CLASS = {
    class_to_idx["akiec"]: "Actinic Keratoses (AKIEC)",
    class_to_idx["bcc"]:   "Basal Cell Carcinoma (BCC)",
    class_to_idx["bkl"]:   "Benign Keratosis-like Lesions (BKL)",
    class_to_idx["df"]:    "Dermatofibroma (DF)",
    class_to_idx["mel"]:   "Melanoma (MEL)",
    class_to_idx["nv"]:    "Melanocytic Nevi (NV)",
    class_to_idx["vasc"]:  "Vascular Lesions (VASC)",
}
class_names = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]

print(f"Classes: {class_names}")
print(f"Class weights (normalised): { {class_names[i]: f'{class_weights_tensor[i]:.3f}' for i in range(NUM_CLASSES)} }")

In [ ]:
class TabularNet(nn.Module):
    def __init__(self, num_meta_features, num_classes=7):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(num_meta_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.mlp(x)

model = TabularNet(num_meta_features=NUM_META_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)
summary(
    model,
    input_size=(BATCH_SIZE, NUM_META_FEATURES),   # tabular only — no image input
    dtypes=[torch.float32],
    col_names=["input_size", "output_size", "num_params"],
    depth=3
)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(weight=weight, reduction="none")

    def forward(self, inputs, targets):
        ce_loss    = self.ce(inputs, targets)
        pt         = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

In [ ]:
criterion = FocalLoss(weight=class_weights_tensor.to(DEVICE), gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

In [ ]:
def compute_specificity(y_true, y_pred, num_classes):
    cm = confusion_matrix(y_true, y_pred)
    specificities = []
    for i in range(num_classes):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)
        specificities.append(TN / (TN + FP) if (TN + FP) > 0 else 0)
    return np.mean(specificities)

def train_tabular(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, y_true, y_pred = 0.0, [], []

    for meta, labels in loader:
        meta, labels = meta.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(meta)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * meta.size(0)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(torch.argmax(outputs, 1).cpu().numpy())

    return (running_loss / len(y_true),
            accuracy_score(y_true, y_pred),
            f1_score(y_true, y_pred, average="macro", zero_division=0))

def evaluate_tabular(model, loader, criterion, device):
    model.eval()
    running_loss, y_true, y_pred = 0.0, [], []

    with torch.no_grad():
        for meta, labels in loader:
            meta, labels = meta.to(device), labels.to(device)
            outputs = model(meta)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * meta.size(0)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(torch.argmax(outputs, 1).cpu().numpy())

    avg_loss = running_loss / len(y_true)
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    spec = compute_specificity(y_true, y_pred, NUM_CLASSES)
    return avg_loss, acc, f1, spec, y_true, y_pred

In [ ]:
EPOCHS           = 50
best_val_f1      = 0.0
best_model_state = None
best_f1_epoch    = 1

train_losses, val_losses = [], []

print("Starting HAM10000 Tabular-Only Ablation Study...")

epoch_pbar = tqdm(range(EPOCHS), desc="Training Progress")

for epoch in epoch_pbar:
    train_loss, train_acc, train_f1 = train_tabular(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, val_f1, val_spec, _, _ = evaluate_tabular(model, val_loader, criterion, DEVICE)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    scheduler.step(val_f1)

    epoch_pbar.set_postfix({
        'Train F1': f'{train_f1:.4f}',
        'Val F1':   f'{val_f1:.4f}',
        'Val Acc':  f'{val_acc:.4f}',
        'Val Spec': f'{val_spec:.4f}'
    })

    if val_f1 > best_val_f1:
        best_val_f1      = val_f1
        best_f1_epoch    = epoch + 1
        best_model_state = model.state_dict()

if best_model_state:
    model.load_state_dict(best_model_state)
    print("\nLoaded best model state based on Validation F1.")

In [ ]:
def print_full_report(split_name, model, loader, criterion, device):
    loss, acc, f1, spec, y_true, y_pred = evaluate_tabular(model, loader, criterion, device)

    print(f"\n{'='*40}")
    print(f"{split_name.upper()} REPORT")
    print(f"{'='*40}")
    print(f"{split_name} Loss: {loss:.4f}")
    print(f"{split_name} Accuracy: {acc:.4f}")
    print(f"{split_name} Specificity (Macro): {spec:.4f}")

    print(f"\n{split_name} Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0, digits=4))

    macro_prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    macro_rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)

    weighted_prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    weighted_rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    weighted_f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print(f"{split_name} Additional Summary:")
    print(f"Macro Avg     -> Precision: {macro_prec:.4f} | Recall: {macro_rec:.4f} | F1: {f1:.4f}")
    print(f"Weighted Avg  -> Precision: {weighted_prec:.4f} | Recall: {weighted_rec:.4f} | F1: {weighted_f1:.4f}")
    print(f"Specificity   -> {spec:.4f}\n")

    return y_true, y_pred

_, _ = print_full_report("Train",      model, train_loader, criterion, DEVICE)
_, _ = print_full_report("Validation", model, val_loader,   criterion, DEVICE)
y_true_test, y_pred_test = print_full_report("Test", model, test_loader, criterion, DEVICE)


# Confusion Matrix
cm = confusion_matrix(y_true_test, y_pred_test)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="mako_r", vmax=100,
    xticklabels=class_names, yticklabels=class_names,
    square=True, cbar_kws={"shrink": 0.8}
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("HAM10000 Tabular-Only Test Confusion Matrix")
plt.tight_layout()
plt.show()


# ROC Curve
model.eval()
all_targets, all_probs = [], []
with torch.no_grad():
    for meta, labels in test_loader:
        probs = F.softmax(model(meta.to(DEVICE)), dim=1)
        all_targets.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

y_true_bin = label_binarize(all_targets, classes=range(NUM_CLASSES))
all_probs  = np.array(all_probs)

plt.figure(figsize=(10, 8))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], all_probs[:, i])
    plt.plot(fpr, tpr, lw=2, label=f'{class_names[i]} (AUC = {auc(fpr, tpr):.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Guessing')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('HAM10000 Tabular-Only Multi-Class ROC Curve', fontsize=16, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# Loss Curves
epochs_range = range(1, len(train_losses) + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs_range, train_losses, label="Training Loss")
plt.plot(epochs_range, val_losses,   label="Validation Loss")
plt.axvline(best_f1_epoch, color="green", linestyle="--", linewidth=1.2,
            label=f"Best Val F1 Epoch ({best_f1_epoch})")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss (HAM10000 Tabular-Only Ablation)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ── Bootstrap Confidence Intervals on Test Set ───────────────

N_BOOTSTRAP = 1000
rng = np.random.default_rng(42)

y_true_arr = np.array(y_true_test)
y_pred_arr = np.array(y_pred_test)
n_test = len(y_true_arr)

boot = {
    "Accuracy": [],
    "F1 (Macro)": [],
    "Precision (Macro)": [],
    "Recall (Macro)": [],
    "Specificity": []
}

per_class_f1 = {name: [] for name in class_names}


def compute_specificity_boot(y_t, y_p, num_classes):
    cm = confusion_matrix(y_t, y_p, labels=list(range(num_classes)))
    specs = []

    for i in range(num_classes):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)

        specs.append(TN / (TN + FP) if (TN + FP) > 0 else 0.0)

    return float(np.mean(specs))


for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n_test, size=n_test)
    y_t, y_p = y_true_arr[idx], y_pred_arr[idx]

    boot["Accuracy"].append(accuracy_score(y_t, y_p))

    boot["F1 (Macro)"].append(
        f1_score(y_t, y_p, average="macro",
                 labels=list(range(NUM_CLASSES)), zero_division=0)
    )

    boot["Precision (Macro)"].append(
        precision_score(y_t, y_p, average="macro",
                        labels=list(range(NUM_CLASSES)), zero_division=0)
    )

    boot["Recall (Macro)"].append(
        recall_score(y_t, y_p, average="macro",
                     labels=list(range(NUM_CLASSES)), zero_division=0)
    )

    boot["Specificity"].append(
        compute_specificity_boot(y_t, y_p, NUM_CLASSES)
    )

    pc = f1_score(
        y_t, y_p, average=None,
        labels=list(range(NUM_CLASSES)), zero_division=0
    )

    for ci, name in enumerate(class_names):
        per_class_f1[name].append(pc[ci])


def ci95(arr):
    return float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))


# ── Summary Table ────────────────────────────────────────────

W = 70

print("\n" + "=" * W)
print(f"  HAM10000 Tabular-Only — Bootstrap CI (n={N_BOOTSTRAP})")
print("=" * W)
print(f"{'Metric':<22} {'Mean':>7}  {'Std':>7}  {'CI Lower':>10}  {'CI Upper':>10}")
print("=" * W)

for metric, values in boot.items():
    arr = np.array(values)
    lo, hi = ci95(arr)

    print(f"{metric:<22} {arr.mean():>7.4f}  {arr.std():>7.4f}  "
          f"{lo:>10.4f}  {hi:>10.4f}")

print("-" * W)
print(f"\n{'Class':<30} {'Mean':>7}  {'Std':>7}  {'CI Lower':>10}  {'CI Upper':>10}")
print("-" * W)

for name in class_names:
    arr = np.array(per_class_f1[name])
    lo, hi = ci95(arr)

    print(f"{name:<30} {arr.mean():>7.4f}  {arr.std():>7.4f}  "
          f"{lo:>10.4f}  {hi:>10.4f}")

print("=" * W)


# ── Box Plot of Bootstrapped Metric Distributions ────────────

fig, axes = plt.subplots(1, 5, figsize=(18, 6))
fig.subplots_adjust(top=0.82, wspace=0.4)

for ax, (label, values) in zip(axes, boot.items()):
    arr = np.array(values)
    lo, hi = ci95(arr)
    mn, sd = arr.mean(), arr.std()

    ax.boxplot(
        arr, widths=0.5, patch_artist=True,
        boxprops=dict(facecolor="#4c72b0", alpha=0.75),
        medianprops=dict(color="white", linewidth=2),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
        flierprops=dict(marker="o", markersize=3,
                        markerfacecolor="gray", alpha=0.5)
    )

    ax.set_title(label, fontsize=10, fontweight="bold", pad=8)
    ax.set_xticks([])
    ax.set_ylabel("Score", fontsize=9)
    ax.grid(axis="y", alpha=0.3, linestyle="--")

    ax.text(
        1, ax.get_ylim()[1],
        f"μ={mn:.4f}\nσ={sd:.4f}\n95% CI [{lo:.4f}, {hi:.4f}]",
        ha="center", va="top", fontsize=7.5, color="#222222",
        bbox=dict(boxstyle="round,pad=0.3",
                  facecolor="lightyellow",
                  edgecolor="gray", alpha=0.8)
    )

plt.suptitle(
    f"HAM10000 Tabular-Only — Bootstrap Metric Distributions "
    f"(n={N_BOOTSTRAP}) — Test Set",
    fontsize=13, fontweight="bold", y=0.97
)

plt.tight_layout()
plt.savefig(
    "/kaggle/working/ham10000_tabular_bootstrap_boxplots.png",
    dpi=150, bbox_inches="tight"
)
plt.show()


# ── Per-Class F1 Bootstrap Bar Chart ─────────────────────────

means = np.array([np.mean(per_class_f1[n]) for n in class_names])
stds = np.array([np.std(per_class_f1[n]) for n in class_names])

lo_arr = np.array([ci95(per_class_f1[n])[0] for n in class_names])
hi_arr = np.array([ci95(per_class_f1[n])[1] for n in class_names])

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(NUM_CLASSES)

bars = ax.bar(
    x, means,
    yerr=[means - lo_arr, hi_arr - means],
    capsize=6,
    color="#4c72b0",
    alpha=0.8,
    ecolor="#333333",
    error_kw={"linewidth": 1.8, "capthick": 1.8},
    width=0.55
)

ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=25, ha="right", fontsize=9)
ax.set_ylabel("F1 Score", fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title(
    "HAM10000 Tabular-Only — Per-Class F1 Mean ± 95% CI (Bootstrap)",
    fontsize=13, fontweight="bold"
)
ax.grid(axis="y", alpha=0.3, linestyle="--")
ax.spines[["top", "right"]].set_visible(False)

for bar, m, s, hi in zip(bars, means, stds, hi_arr):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        hi + 0.025,
        f"{m:.3f}\n±{s:.3f}",
        ha="center", va="bottom", fontsize=9, color="#111111"
    )

plt.tight_layout()
plt.savefig(
    "/kaggle/working/ham10000_tabular_perclass_f1.png",
    dpi=150, bbox_inches="tight"
)
plt.show()